# March ML Mania 2026 - Expert Models & Specialized Architectures
**TabNet + Bradley-Terry + Gaussian Process + SVM + Random Forest + Wide&Deep + KNN**

This notebook adds specialized models that complement the existing GBM and DL pipelines.
All models feed into a mega-ensemble.

## 0. Setup & Install

In [ ]:
import os
IS_KAGGLE = os.path.exists("/kaggle/input")
if IS_KAGGLE:
    os.system("pip install -q pytorch-tabnet 2>/dev/null")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.optimize import minimize
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.linear_model import LogisticRegression, Ridge, BayesianRidge
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

CLIP_MIN, CLIP_MAX = 0.05, 0.95

if IS_KAGGLE:
    DATA_DIR = Path("/kaggle/input/competitions/march-machine-learning-mania-2026")
    OUT_DIR = Path("/kaggle/working")
else:
    DATA_DIR = Path(__file__).parent.parent / "data" / "raw"
    OUT_DIR = Path(__file__).parent.parent / ".tmp"
OUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Data Loading & Feature Engineering (Shared Pipeline)

In [ ]:
# Load data
m_reg_compact = pd.read_csv(DATA_DIR / "MRegularSeasonCompactResults.csv")
m_reg_detailed = pd.read_csv(DATA_DIR / "MRegularSeasonDetailedResults.csv")
m_tourney_compact = pd.read_csv(DATA_DIR / "MNCAATourneyCompactResults.csv")
m_seeds = pd.read_csv(DATA_DIR / "MNCAATourneySeeds.csv")
m_massey = pd.read_csv(DATA_DIR / "MMasseyOrdinals.csv")
m_teams = pd.read_csv(DATA_DIR / "MTeams.csv")
m_coaches = pd.read_csv(DATA_DIR / "MTeamCoaches.csv")
m_conf_tourney = pd.read_csv(DATA_DIR / "MConferenceTourneyGames.csv")
m_conferences = pd.read_csv(DATA_DIR / "MTeamConferences.csv")

w_reg_compact = pd.read_csv(DATA_DIR / "WRegularSeasonCompactResults.csv")
w_reg_detailed = pd.read_csv(DATA_DIR / "WRegularSeasonDetailedResults.csv")
w_tourney_compact = pd.read_csv(DATA_DIR / "WNCAATourneyCompactResults.csv")
w_seeds = pd.read_csv(DATA_DIR / "WNCAATourneySeeds.csv")

sub1 = pd.read_csv(DATA_DIR / "SampleSubmissionStage1.csv")
sub2 = pd.read_csv(DATA_DIR / "SampleSubmissionStage2.csv")

m_seeds['SeedNum'] = m_seeds['Seed'].str[1:3].astype(int)
w_seeds['SeedNum'] = w_seeds['Seed'].str[1:3].astype(int)
print("Data loaded.")

In [ ]:
# ========== ELO SYSTEM ==========
class EloSystem:
    def __init__(self, k=32, home_adv=100, margin_mult=0.006, reversion=0.25):
        self.k, self.home_adv, self.margin_mult, self.reversion = k, home_adv, margin_mult, reversion
        self.ratings, self.initial = {}, 1500

    def get(self, t): return self.ratings.get(t, self.initial)
    def expected(self, ra, rb): return 1.0 / (1.0 + 10.0 ** ((rb - ra) / 400.0))

    def update(self, w, l, margin, wloc='N'):
        rw, rl = self.get(w), self.get(l)
        rw_a = rw + (self.home_adv if wloc == 'H' else 0)
        rl_a = rl + (self.home_adv if wloc == 'A' else 0)
        exp_w = self.expected(rw_a, rl_a)
        mov = np.log(abs(margin) + 1) * (2.2 / (abs(rw - rl) * self.margin_mult + 2.2))
        adj = self.k * mov * (1 - exp_w)
        self.ratings[w], self.ratings[l] = rw + adj, rl - adj

    def new_season(self):
        for t in self.ratings:
            self.ratings[t] = self.ratings[t] * (1 - self.reversion) + self.initial * self.reversion

def build_elo(reg_df, tourney_df=None, k=32):
    elo = EloSystem(k=k)
    all_g = pd.concat([reg_df] + ([tourney_df] if tourney_df is not None else []), ignore_index=True)
    all_g = all_g.sort_values(['Season', 'DayNum']).reset_index(drop=True)
    season_ratings, prev = {}, None
    for _, g in all_g.iterrows():
        if g['Season'] != prev:
            if prev is not None: elo.new_season()
            prev = g['Season']
        elo.update(g['WTeamID'], g['LTeamID'], g['WScore'] - g['LScore'], g.get('WLoc', 'N'))
        if 132 <= g['DayNum'] <= 133:
            season_ratings[g['Season']] = dict(elo.ratings)
    season_ratings[all_g['Season'].max()] = dict(elo.ratings)
    rows = [{'Season': s, 'TeamID': t, 'EloRating': r} for s, rats in season_ratings.items() for t, r in rats.items()]
    return pd.DataFrame(rows), elo

print("Building Elo...")
m_elo_df, m_elo = build_elo(m_reg_compact, m_tourney_compact, k=32)
w_elo_df, w_elo = build_elo(w_reg_compact, w_tourney_compact, k=32)

In [ ]:
# ========== TEAM SEASON STATS ==========
def compute_team_stats(det_df, comp_df):
    def extract(df, p):
        o = 'L' if p == 'W' else 'W'
        r = pd.DataFrame({'Season': df['Season'], 'TeamID': df[f'{p}TeamID'], 'DayNum': df['DayNum'],
                          'Win': 1 if p == 'W' else 0, 'Score': df[f'{p}Score'], 'OppScore': df[f'{o}Score'],
                          'FGM': df[f'{p}FGM'], 'FGA': df[f'{p}FGA'], 'FGM3': df[f'{p}FGM3'], 'FGA3': df[f'{p}FGA3'],
                          'FTM': df[f'{p}FTM'], 'FTA': df[f'{p}FTA'], 'OR': df[f'{p}OR'], 'DR': df[f'{p}DR'],
                          'Ast': df[f'{p}Ast'], 'TO': df[f'{p}TO'], 'Stl': df[f'{p}Stl'], 'Blk': df[f'{p}Blk'],
                          'OppOR': df[f'{o}OR'], 'OppDR': df[f'{o}DR'], 'OppFGA': df[f'{o}FGA'],
                          'OppFTA': df[f'{o}FTA'], 'OppTO': df[f'{o}TO'], 'OppFGM': df[f'{o}FGM'],
                          'OppFGM3': df[f'{o}FGM3']})
        return r

    all_g = pd.concat([extract(det_df, 'W'), extract(det_df, 'L')], ignore_index=True)
    reg = all_g[all_g['DayNum'] < 132]

    agg = reg.groupby(['Season', 'TeamID']).agg({
        'Win': ['sum', 'count'], 'Score': 'mean', 'OppScore': 'mean',
        'FGM': 'mean', 'FGA': 'mean', 'FGM3': 'mean', 'FGA3': 'mean',
        'FTM': 'mean', 'FTA': 'mean', 'OR': 'mean', 'DR': 'mean',
        'Ast': 'mean', 'TO': 'mean', 'Stl': 'mean', 'Blk': 'mean',
        'OppOR': 'mean', 'OppDR': 'mean', 'OppFGA': 'mean', 'OppFTA': 'mean',
        'OppTO': 'mean', 'OppFGM': 'mean', 'OppFGM3': 'mean'
    }).reset_index()
    agg.columns = ['Season', 'TeamID', 'Wins', 'Games', 'Score', 'OppScore',
                    'FGM', 'FGA', 'FGM3', 'FGA3', 'FTM', 'FTA', 'OR', 'DR',
                    'Ast', 'TO', 'Stl', 'Blk', 'OppOR', 'OppDR', 'OppFGA',
                    'OppFTA', 'OppTO', 'OppFGM', 'OppFGM3']

    agg['WinPct'] = agg['Wins'] / agg['Games']
    agg['PointDiff'] = agg['Score'] - agg['OppScore']
    agg['eFG_pct'] = (agg['FGM'] + 0.5 * agg['FGM3']) / agg['FGA']
    poss = agg['FGA'] + 0.44 * agg['FTA'] + agg['TO']
    agg['TO_pct'] = agg['TO'] / poss
    agg['ORB_pct'] = agg['OR'] / (agg['OR'] + agg['OppDR'])
    agg['FT_rate'] = agg['FTM'] / agg['FGA']
    agg['Opp_eFG_pct'] = (agg['OppFGM'] + 0.5 * agg['OppFGM3']) / agg['OppFGA']
    opp_poss = agg['OppFGA'] + 0.44 * agg['OppFTA'] + agg['OppTO']
    agg['OffRating'] = agg['Score'] / poss * 100
    agg['DefRating'] = agg['OppScore'] / opp_poss * 100
    agg['NetRating'] = agg['OffRating'] - agg['DefRating']
    agg['Pace'] = (poss + opp_poss) / 2
    agg['FG3_pct'] = agg['FGM3'] / agg['FGA3']
    agg['FT_pct'] = agg['FTM'] / agg['FTA']
    agg['Ast_TO'] = agg['Ast'] / agg['TO']
    agg['Opp_TO_pct'] = agg['OppTO'] / opp_poss

    # Last 10
    l10 = reg.sort_values('DayNum').groupby(['Season', 'TeamID']).tail(10)
    l10a = l10.groupby(['Season', 'TeamID']).agg({'Win': 'mean', 'Score': 'mean', 'OppScore': 'mean'}).reset_index()
    l10a.columns = ['Season', 'TeamID', 'L10_WinPct', 'L10_Score', 'L10_OppScore']
    l10a['L10_PointDiff'] = l10a['L10_Score'] - l10a['L10_OppScore']
    agg = agg.merge(l10a, on=['Season', 'TeamID'], how='left')

    # Consistency
    gm = reg.copy(); gm['Margin'] = gm['Score'] - gm['OppScore']
    cons = gm.groupby(['Season', 'TeamID'])['Margin'].std().reset_index(name='MarginStd')
    agg = agg.merge(cons, on=['Season', 'TeamID'], how='left')

    # Road performance
    road_games = all_g.copy()
    road_w = road_games[(road_games['DayNum'] < 132)]
    road_w['IsRoad'] = 0  # Simplified: we'll compute from compact results
    road_agg = road_w.groupby(['Season', 'TeamID'])['Win'].agg(['sum', 'count']).reset_index()
    road_agg.columns = ['Season', 'TeamID', 'RoadWins', 'RoadGames']
    road_agg['RoadWinPct'] = road_agg['RoadWins'] / road_agg['RoadGames']

    return agg

print("Computing team stats...")
m_stats = compute_team_stats(m_reg_detailed, m_reg_compact)
w_stats = compute_team_stats(w_reg_detailed, w_reg_compact)

In [ ]:
# ========== MASSEY ORDINALS ==========
TOP_SYS = ['POM', 'SAG', 'MOR', 'DOL', 'COL', 'RPI']
eos = m_massey[(m_massey['RankingDayNum'] >= 128) & (m_massey['RankingDayNum'] <= 133) &
               (m_massey['SystemName'].isin(TOP_SYS))]
eos = eos.sort_values('RankingDayNum').groupby(['Season', 'SystemName', 'TeamID']).tail(1)
m_massey_feat = eos.pivot_table(index=['Season', 'TeamID'], columns='SystemName',
                                 values='OrdinalRank', aggfunc='first').reset_index()
rank_cols = [c for c in m_massey_feat.columns if c in TOP_SYS]
m_massey_feat['ConsensusRank'] = m_massey_feat[rank_cols].mean(axis=1)
print(f"Massey features: {m_massey_feat.shape}")

In [ ]:
# ========== COACH EXPERIENCE ==========
def compute_coach_features(coaches_df, tourney_df):
    """Coach tournament experience as cumulative prior wins."""
    coach_season = coaches_df.copy()
    coach_season['CoachName'] = coach_season['CoachName'].str.strip()
    # Get last coach for each team-season
    coach_last = coach_season.sort_values('LastDayNum').groupby(['Season', 'TeamID']).tail(1)[['Season', 'TeamID', 'CoachName']]

    # Count tourney games won by coach across prior seasons
    tourney_coaches = tourney_df.merge(coach_last, left_on=['Season', 'WTeamID'], right_on=['Season', 'TeamID'], how='left')
    coach_wins = tourney_coaches.groupby(['CoachName', 'Season']).size().reset_index(name='TourneyWins')
    coach_wins = coach_wins.sort_values(['CoachName', 'Season'])
    coach_wins['CumWins'] = coach_wins.groupby('CoachName')['TourneyWins'].cumsum().shift(1).fillna(0)

    # Merge back
    coach_exp = coach_last.merge(coach_wins[['CoachName', 'Season', 'CumWins']],
                                  on=['CoachName', 'Season'], how='left')
    coach_exp['CoachTourneyWins'] = coach_exp['CumWins'].fillna(0)
    return coach_exp[['Season', 'TeamID', 'CoachTourneyWins']]

m_coach_feat = compute_coach_features(m_coaches, m_tourney_compact)
print(f"Coach features: {m_coach_feat.shape}")

In [ ]:
# ========== CONFERENCE TOURNAMENT ==========
def compute_conf_tourney_features(conf_tourney_df):
    """Conference tournament performance features."""
    rows = []
    for p in ['W', 'L']:
        df = conf_tourney_df.copy()
        rows.append(pd.DataFrame({
            'Season': df['Season'], 'TeamID': df[f'{p}TeamID'],
            'Win': 1 if p == 'W' else 0
        }))
    all_ct = pd.concat(rows, ignore_index=True)
    agg = all_ct.groupby(['Season', 'TeamID']).agg(
        ConfTourneyWins=('Win', 'sum'),
        ConfTourneyGames=('Win', 'count')
    ).reset_index()
    # Did team win the conference tournament? (won last game)
    champs = conf_tourney_df.groupby(['Season']).apply(
        lambda x: x.sort_values('DayNum').groupby(
            conf_tourney_df.columns[conf_tourney_df.columns.str.contains('Conf')].tolist()[0] if 'ConfAbbrev' in conf_tourney_df.columns else 'Season'
        ).tail(1)['WTeamID'].values, include_groups=False
    )
    return agg

try:
    m_conf_feat = compute_conf_tourney_features(m_conf_tourney)
    print(f"Conf tourney features: {m_conf_feat.shape}")
    HAS_CONF = True
except Exception:
    HAS_CONF = False
    print("Conf tourney features: skipped")

In [ ]:
# ========== TEAM FEATURE VECTOR ==========
TEAM_FEATURES = ['WinPct', 'PointDiff', 'eFG_pct', 'TO_pct', 'ORB_pct', 'FT_rate',
                  'OffRating', 'DefRating', 'NetRating', 'Pace', 'FG3_pct', 'FT_pct',
                  'Ast_TO', 'Opp_eFG_pct', 'L10_WinPct', 'L10_PointDiff', 'MarginStd',
                  'Score', 'OppScore', 'Stl', 'Blk']

def get_team_vector(stats_df, elo_df, season, team_id):
    row = stats_df[(stats_df['Season'] == season) & (stats_df['TeamID'] == team_id)]
    if len(row) == 0:
        return None
    r = row.iloc[0]
    feats = [r.get(f, 0) for f in TEAM_FEATURES]
    elo_row = elo_df[(elo_df['Season'] == season) & (elo_df['TeamID'] == team_id)]
    feats.append(elo_row.iloc[0]['EloRating'] if len(elo_row) > 0 else 1500)
    return np.array(feats, dtype=np.float32)

N_TEAM_FEATURES = len(TEAM_FEATURES) + 1
print(f"Team feature vector size: {N_TEAM_FEATURES}")

## 2. Build Training Data (Enhanced with Extra Features)

In [ ]:
def build_all_training_data(tourney_df, seeds_df, stats_df, elo_df, massey_df=None,
                             coach_df=None, conf_df=None):
    """Build training data with enhanced feature set."""
    tabular_rows = []
    team_a_vectors = []
    team_b_vectors = []
    targets = []
    meta = []

    for _, game in tourney_df.iterrows():
        season = game['Season']
        w_id, l_id = game['WTeamID'], game['LTeamID']
        team_a, team_b = min(w_id, l_id), max(w_id, l_id)
        target = 1 if team_a == w_id else 0

        vec_a = get_team_vector(stats_df, elo_df, season, team_a)
        vec_b = get_team_vector(stats_df, elo_df, season, team_b)
        if vec_a is None or vec_b is None:
            continue

        # Seeds
        sa = seeds_df[(seeds_df['Season'] == season) & (seeds_df['TeamID'] == team_a)]
        sb = seeds_df[(seeds_df['Season'] == season) & (seeds_df['TeamID'] == team_b)]
        if len(sa) == 0 or len(sb) == 0:
            continue
        seed_a, seed_b = sa.iloc[0]['SeedNum'], sb.iloc[0]['SeedNum']

        # Tabular: difference features
        diff = vec_a - vec_b
        tab_feat = np.concatenate([diff, [seed_a - seed_b, seed_a, seed_b]])

        # Massey
        massey_feats = []
        if massey_df is not None:
            am = massey_df[(massey_df['Season'] == season) & (massey_df['TeamID'] == team_a)]
            bm = massey_df[(massey_df['Season'] == season) & (massey_df['TeamID'] == team_b)]
            for sys in ['POM', 'SAG', 'MOR', 'ConsensusRank']:
                if len(am) > 0 and len(bm) > 0 and sys in am.columns:
                    va_val = am.iloc[0][sys] if not pd.isna(am.iloc[0].get(sys)) else 150
                    vb_val = bm.iloc[0][sys] if not pd.isna(bm.iloc[0].get(sys)) else 150
                    massey_feats.append(va_val - vb_val)
                else:
                    massey_feats.append(0)
        tab_feat = np.concatenate([tab_feat, massey_feats])

        # Coach experience
        coach_feats = []
        if coach_df is not None:
            ca = coach_df[(coach_df['Season'] == season) & (coach_df['TeamID'] == team_a)]
            cb = coach_df[(coach_df['Season'] == season) & (coach_df['TeamID'] == team_b)]
            cw_a = ca.iloc[0]['CoachTourneyWins'] if len(ca) > 0 else 0
            cw_b = cb.iloc[0]['CoachTourneyWins'] if len(cb) > 0 else 0
            coach_feats = [cw_a - cw_b]
        tab_feat = np.concatenate([tab_feat, coach_feats])

        # Conf tourney
        conf_feats = []
        if conf_df is not None:
            cta = conf_df[(conf_df['Season'] == season) & (conf_df['TeamID'] == team_a)]
            ctb = conf_df[(conf_df['Season'] == season) & (conf_df['TeamID'] == team_b)]
            ctw_a = cta.iloc[0]['ConfTourneyWins'] if len(cta) > 0 else 0
            ctw_b = ctb.iloc[0]['ConfTourneyWins'] if len(ctb) > 0 else 0
            conf_feats = [ctw_a - ctw_b]
        tab_feat = np.concatenate([tab_feat, conf_feats])

        # Interactions
        seed_diff = seed_a - seed_b
        elo_diff = vec_a[-1] - vec_b[-1]
        net_diff = diff[TEAM_FEATURES.index('NetRating')] if 'NetRating' in TEAM_FEATURES else 0
        tab_feat = np.concatenate([tab_feat, [seed_diff * elo_diff, seed_diff * net_diff]])

        # Quadratic seed feature (non-linear seed effect)
        tab_feat = np.concatenate([tab_feat, [seed_diff ** 2, abs(seed_diff)]])

        tabular_rows.append(tab_feat)
        team_a_vectors.append(vec_a)
        team_b_vectors.append(vec_b)
        targets.append(target)
        meta.append({'Season': season, 'TeamA': team_a, 'TeamB': team_b,
                     'SeedA': seed_a, 'SeedB': seed_b})

    return (np.array(tabular_rows, dtype=np.float32),
            np.array(team_a_vectors, dtype=np.float32),
            np.array(team_b_vectors, dtype=np.float32),
            np.array(targets, dtype=np.float32),
            pd.DataFrame(meta))

print("Building training data...")
m_tab, m_vec_a, m_vec_b, m_y, m_meta = build_all_training_data(
    m_tourney_compact, m_seeds, m_stats, m_elo_df, m_massey_feat,
    m_coach_feat if 'm_coach_feat' in dir() else None,
    m_conf_feat if HAS_CONF else None)
w_tab, w_vec_a, w_vec_b, w_y, w_meta = build_all_training_data(
    w_tourney_compact, w_seeds, w_stats, w_elo_df)

# Combine
tab_all = np.vstack([m_tab, np.pad(w_tab, ((0,0),(0, m_tab.shape[1] - w_tab.shape[1])))])
vec_a_all = np.vstack([m_vec_a, w_vec_a])
vec_b_all = np.vstack([m_vec_b, w_vec_b])
y_all = np.concatenate([m_y, w_y])
meta_all = pd.concat([m_meta, w_meta], ignore_index=True)
seasons_all = meta_all['Season'].values

# Handle NaN
tab_all = np.nan_to_num(tab_all, nan=0.0)
vec_a_all = np.nan_to_num(vec_a_all, nan=0.0)
vec_b_all = np.nan_to_num(vec_b_all, nan=0.0)

# Scale
tab_scaler = StandardScaler()
tab_scaled = tab_scaler.fit_transform(tab_all)
vec_scaler = StandardScaler()
all_vecs = np.vstack([vec_a_all, vec_b_all])
vec_scaler.fit(all_vecs)
vec_a_scaled = vec_scaler.transform(vec_a_all)
vec_b_scaled = vec_scaler.transform(vec_b_all)

N_TAB_FEATURES = tab_all.shape[1]
print(f"Training: {len(y_all)} games, {N_TAB_FEATURES} tabular features, {N_TEAM_FEATURES} team features")

## 3. Expert Model Architectures

### 3.1 Bradley-Terry Model
Classic pairwise comparison model from sports analytics.
Models P(A beats B) = strength_A / (strength_A + strength_B)

In [ ]:
class BradleyTerryModel:
    """Bradley-Terry model with regularization. Learns team strengths from game outcomes."""
    def __init__(self, alpha=1.0, max_iter=200, lr=0.01):
        self.alpha = alpha
        self.max_iter = max_iter
        self.lr = lr
        self.strengths = {}

    def fit(self, games_df, y=None):
        """
        games_df must have: Season, WTeamID, LTeamID
        Or pass separate arrays with team_a, team_b, result.
        """
        # Initialize strengths
        all_teams = set(games_df['WTeamID'].values) | set(games_df['LTeamID'].values)
        self.strengths = {t: 0.0 for t in all_teams}

        for iteration in range(self.max_iter):
            grad = {t: 0.0 for t in all_teams}
            total_loss = 0

            for _, g in games_df.iterrows():
                w, l = g['WTeamID'], g['LTeamID']
                sw, sl = self.strengths[w], self.strengths[l]
                p_w = 1.0 / (1.0 + np.exp(sl - sw))  # P(W wins)

                # Gradient: winner gets (1-p), loser gets -(1-p)
                grad[w] += (1.0 - p_w)
                grad[l] -= (1.0 - p_w)
                total_loss += -np.log(max(p_w, 1e-10))

            # L2 regularization
            for t in all_teams:
                grad[t] -= self.alpha * self.strengths[t]

            # Update
            for t in all_teams:
                self.strengths[t] += self.lr * grad[t]

            # Normalize (fix identifiability)
            mean_s = np.mean(list(self.strengths.values()))
            for t in all_teams:
                self.strengths[t] -= mean_s

    def predict(self, team_a, team_b):
        """P(team_a beats team_b)"""
        sa = self.strengths.get(team_a, 0.0)
        sb = self.strengths.get(team_b, 0.0)
        return 1.0 / (1.0 + np.exp(sb - sa))

    def get_strength_diff(self, team_a, team_b):
        return self.strengths.get(team_a, 0.0) - self.strengths.get(team_b, 0.0)


def bt_cv(reg_compact, tourney_compact, seeds_df, meta_df, y, seasons):
    """Bradley-Terry with leave-one-season-out CV."""
    val_seasons = sorted(set(s for s in np.unique(seasons) if s >= 2015))
    oof = np.full(len(y), np.nan)

    for vs in val_seasons:
        # Train BT on all regular season + tourney games before val_season
        train_games = reg_compact[reg_compact['Season'] < vs]
        train_tourney = tourney_compact[tourney_compact['Season'] < vs]
        all_train = pd.concat([train_games, train_tourney], ignore_index=True)

        bt = BradleyTerryModel(alpha=0.5, max_iter=100, lr=0.02)
        bt.fit(all_train)

        # Also train on current season regular games
        curr_reg = reg_compact[reg_compact['Season'] == vs]
        if len(curr_reg) > 0:
            bt_curr = BradleyTerryModel(alpha=0.5, max_iter=100, lr=0.02)
            bt_curr.fit(pd.concat([all_train, curr_reg], ignore_index=True))
        else:
            bt_curr = bt

        va_mask = seasons == vs
        for i in np.where(va_mask)[0]:
            ta, tb = int(meta_df.iloc[i]['TeamA']), int(meta_df.iloc[i]['TeamB'])
            pred = bt_curr.predict(ta, tb)
            oof[i] = np.clip(pred, CLIP_MIN, CLIP_MAX)

    valid = ~np.isnan(oof)
    bs = np.mean((y[valid] - oof[valid]) ** 2)
    print(f"  Bradley-Terry Brier: {bs:.4f}")
    return oof, valid, bs

# We need separate BT for men/women
# For now, use combined data through the meta approach
print("\n" + "=" * 60)
print("MODEL 1: BRADLEY-TERRY")
print("=" * 60)

# Build BT predictions using meta info
bt_oof = np.full(len(y_all), np.nan)
n_mens = len(m_y)

# Men's BT
print("  Men's BT...")
m_val_seasons = sorted(set(s for s in m_meta['Season'].unique() if s >= 2015))
for vs in m_val_seasons:
    all_train = pd.concat([
        m_reg_compact[m_reg_compact['Season'] <= vs],
        m_tourney_compact[m_tourney_compact['Season'] < vs]
    ], ignore_index=True)
    bt = BradleyTerryModel(alpha=0.5, max_iter=150, lr=0.02)
    bt.fit(all_train)

    va_mask = (m_meta['Season'] == vs).values
    for i in np.where(va_mask)[0]:
        ta, tb = int(m_meta.iloc[i]['TeamA']), int(m_meta.iloc[i]['TeamB'])
        bt_oof[i] = np.clip(bt.predict(ta, tb), CLIP_MIN, CLIP_MAX)

# Women's BT
print("  Women's BT...")
w_val_seasons = sorted(set(s for s in w_meta['Season'].unique() if s >= 2015))
for vs in w_val_seasons:
    all_train = pd.concat([
        w_reg_compact[w_reg_compact['Season'] <= vs],
        w_tourney_compact[w_tourney_compact['Season'] < vs]
    ], ignore_index=True)
    bt = BradleyTerryModel(alpha=0.5, max_iter=150, lr=0.02)
    bt.fit(all_train)

    va_mask = (w_meta['Season'] == vs).values
    for i in np.where(va_mask)[0]:
        ta, tb = int(w_meta.iloc[i]['TeamA']), int(w_meta.iloc[i]['TeamB'])
        bt_oof[n_mens + i] = np.clip(bt.predict(ta, tb), CLIP_MIN, CLIP_MAX)

bt_valid = ~np.isnan(bt_oof)
if bt_valid.sum() > 0:
    bt_brier = np.mean((y_all[bt_valid] - bt_oof[bt_valid]) ** 2)
    print(f"  Bradley-Terry Overall Brier: {bt_brier:.4f}")
else:
    bt_brier = 0.25
    print("  Bradley-Terry: no valid predictions")

### 3.2 Random Forest (Diversity Model)

In [ ]:
print("\n" + "=" * 60)
print("MODEL 2: RANDOM FOREST")
print("=" * 60)

def rf_cv(X, y, seasons, name="RF"):
    val_seasons = sorted(set(s for s in np.unique(seasons) if s >= 2015))
    oof = np.full(len(y), np.nan)
    for vs in val_seasons:
        tr, va = seasons < vs, seasons == vs
        if va.sum() == 0: continue
        m = RandomForestClassifier(n_estimators=500, max_depth=8, min_samples_leaf=10,
                                     max_features='sqrt', random_state=SEED, n_jobs=-1)
        m.fit(X[tr], y[tr])
        oof[va] = np.clip(m.predict_proba(X[va])[:, 1], CLIP_MIN, CLIP_MAX)
    valid = ~np.isnan(oof)
    bs = np.mean((y[valid] - oof[valid]) ** 2)
    print(f"  {name} Brier: {bs:.4f}")
    return oof, valid, bs

oof_rf, vm_rf, bs_rf = rf_cv(tab_all, y_all, seasons_all, "Random Forest")

### 3.3 Extra Trees (Maximum Diversity)

In [ ]:
print("\n" + "=" * 60)
print("MODEL 3: EXTRA TREES")
print("=" * 60)

def et_cv(X, y, seasons, name="ET"):
    val_seasons = sorted(set(s for s in np.unique(seasons) if s >= 2015))
    oof = np.full(len(y), np.nan)
    for vs in val_seasons:
        tr, va = seasons < vs, seasons == vs
        if va.sum() == 0: continue
        m = ExtraTreesClassifier(n_estimators=500, max_depth=10, min_samples_leaf=8,
                                   max_features='sqrt', random_state=SEED, n_jobs=-1)
        m.fit(X[tr], y[tr])
        oof[va] = np.clip(m.predict_proba(X[va])[:, 1], CLIP_MIN, CLIP_MAX)
    valid = ~np.isnan(oof)
    bs = np.mean((y[valid] - oof[valid]) ** 2)
    print(f"  {name} Brier: {bs:.4f}")
    return oof, valid, bs

oof_et, vm_et, bs_et = et_cv(tab_all, y_all, seasons_all, "Extra Trees")

### 3.4 SVM with RBF Kernel (Probability Calibrated)

In [ ]:
print("\n" + "=" * 60)
print("MODEL 4: SVM (RBF)")
print("=" * 60)

def svm_cv(X, y, seasons, name="SVM"):
    val_seasons = sorted(set(s for s in np.unique(seasons) if s >= 2015))
    oof = np.full(len(y), np.nan)
    for vs in val_seasons:
        tr, va = seasons < vs, seasons == vs
        if va.sum() == 0: continue
        svm = SVC(C=1.0, kernel='rbf', gamma='scale', probability=True, random_state=SEED)
        svm.fit(X[tr], y[tr])
        oof[va] = np.clip(svm.predict_proba(X[va])[:, 1], CLIP_MIN, CLIP_MAX)
    valid = ~np.isnan(oof)
    bs = np.mean((y[valid] - oof[valid]) ** 2)
    print(f"  {name} Brier: {bs:.4f}")
    return oof, valid, bs

oof_svm, vm_svm, bs_svm = svm_cv(tab_scaled, y_all, seasons_all, "SVM-RBF")

### 3.5 K-Nearest Neighbors (Instance-Based)

In [ ]:
print("\n" + "=" * 60)
print("MODEL 5: KNN")
print("=" * 60)

def knn_cv(X, y, seasons, name="KNN"):
    val_seasons = sorted(set(s for s in np.unique(seasons) if s >= 2015))
    oof = np.full(len(y), np.nan)
    for vs in val_seasons:
        tr, va = seasons < vs, seasons == vs
        if va.sum() == 0: continue
        m = KNeighborsClassifier(n_neighbors=50, weights='distance', metric='minkowski', p=2)
        m.fit(X[tr], y[tr])
        oof[va] = np.clip(m.predict_proba(X[va])[:, 1], CLIP_MIN, CLIP_MAX)
    valid = ~np.isnan(oof)
    bs = np.mean((y[valid] - oof[valid]) ** 2)
    print(f"  {name} Brier: {bs:.4f}")
    return oof, valid, bs

oof_knn, vm_knn, bs_knn = knn_cv(tab_scaled, y_all, seasons_all, "KNN-50")

### 3.6 Gradient Boosting (sklearn native - different from XGBoost/LightGBM)

In [ ]:
print("\n" + "=" * 60)
print("MODEL 6: GRADIENT BOOSTING (sklearn)")
print("=" * 60)

def gbc_cv(X, y, seasons, name="GBC"):
    val_seasons = sorted(set(s for s in np.unique(seasons) if s >= 2015))
    oof = np.full(len(y), np.nan)
    for vs in val_seasons:
        tr, va = seasons < vs, seasons == vs
        if va.sum() == 0: continue
        m = GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=4,
                                         subsample=0.8, min_samples_leaf=10,
                                         random_state=SEED)
        m.fit(X[tr], y[tr])
        oof[va] = np.clip(m.predict_proba(X[va])[:, 1], CLIP_MIN, CLIP_MAX)
    valid = ~np.isnan(oof)
    bs = np.mean((y[valid] - oof[valid]) ** 2)
    print(f"  {name} Brier: {bs:.4f}")
    return oof, valid, bs

oof_gbc, vm_gbc, bs_gbc = gbc_cv(tab_all, y_all, seasons_all, "GradientBoosting")

### 3.7 Bayesian Ridge (Probabilistic Linear Model)

In [ ]:
print("\n" + "=" * 60)
print("MODEL 7: BAYESIAN RIDGE")
print("=" * 60)

def bayesian_cv(X, y, seasons, name="BayesRidge"):
    val_seasons = sorted(set(s for s in np.unique(seasons) if s >= 2015))
    oof = np.full(len(y), np.nan)
    for vs in val_seasons:
        tr, va = seasons < vs, seasons == vs
        if va.sum() == 0: continue
        m = BayesianRidge(max_iter=300, tol=1e-6)
        m.fit(X[tr], y[tr])
        pred = m.predict(X[va])
        oof[va] = np.clip(pred, CLIP_MIN, CLIP_MAX)
    valid = ~np.isnan(oof)
    bs = np.mean((y[valid] - oof[valid]) ** 2)
    print(f"  {name} Brier: {bs:.4f}")
    return oof, valid, bs

oof_bayes, vm_bayes, bs_bayes = bayesian_cv(tab_scaled, y_all, seasons_all, "BayesianRidge")

### 3.8 TabNet (Attention-Based Tabular DL)

In [ ]:
print("\n" + "=" * 60)
print("MODEL 8: TABNET")
print("=" * 60)

try:
    from pytorch_tabnet.tab_model import TabNetClassifier

    def tabnet_cv(X, y, seasons, name="TabNet"):
        val_seasons = sorted(set(s for s in np.unique(seasons) if s >= 2015))
        oof = np.full(len(y), np.nan)
        for vs in val_seasons:
            tr, va = seasons < vs, seasons == vs
            if va.sum() == 0: continue
            m = TabNetClassifier(
                n_d=16, n_a=16, n_steps=3,
                gamma=1.5, lambda_sparse=1e-3,
                optimizer_fn=torch.optim.Adam,
                optimizer_params=dict(lr=2e-2),
                scheduler_params={"step_size": 15, "gamma": 0.9},
                scheduler_fn=torch.optim.lr_scheduler.StepLR,
                mask_type='entmax',
                verbose=0, seed=SEED
            )
            m.fit(
                X[tr], y[tr].astype(int),
                eval_set=[(X[va], y[va].astype(int))],
                eval_metric=['logloss'],
                max_epochs=100, patience=15,
                batch_size=256, virtual_batch_size=128
            )
            oof[va] = np.clip(m.predict_proba(X[va])[:, 1], CLIP_MIN, CLIP_MAX)
        valid = ~np.isnan(oof)
        bs = np.mean((y[valid] - oof[valid]) ** 2)
        print(f"  {name} Brier: {bs:.4f}")
        return oof, valid, bs

    oof_tabnet, vm_tabnet, bs_tabnet = tabnet_cv(tab_all, y_all, seasons_all, "TabNet")
    HAS_TABNET = True
except ImportError:
    HAS_TABNET = False
    print("  TabNet not available (pip install pytorch-tabnet)")

### 3.9 Wide & Deep Network (Google-Style)

In [ ]:
print("\n" + "=" * 60)
print("MODEL 9: WIDE & DEEP")
print("=" * 60)

class WideAndDeep(nn.Module):
    """Wide & Deep model: wide (linear) path + deep (MLP) path."""
    def __init__(self, input_dim, deep_dims=[128, 64, 32]):
        super().__init__()
        # Wide path (linear)
        self.wide = nn.Linear(input_dim, 1)

        # Deep path (MLP with BN)
        layers = []
        prev_dim = input_dim
        for dim in deep_dims:
            layers.extend([
                nn.Linear(prev_dim, dim),
                nn.BatchNorm1d(dim),
                nn.ReLU(),
                nn.Dropout(0.3)
            ])
            prev_dim = dim
        self.deep = nn.Sequential(*layers)
        self.deep_out = nn.Linear(prev_dim, 1)

        # Combine
        self.final = nn.Linear(2, 1)

    def forward(self, x):
        wide_out = self.wide(x)
        deep_out = self.deep_out(self.deep(x))
        combined = torch.cat([wide_out, deep_out], dim=1)
        return torch.sigmoid(self.final(combined)).squeeze(1)

def wide_deep_cv(X, y, seasons, epochs=80, lr=1e-3, batch_size=128, patience=15, name="W&D"):
    val_seasons = sorted(set(s for s in np.unique(seasons) if s >= 2015))
    oof = np.full(len(y), np.nan)

    for vs in val_seasons:
        tr_mask = seasons < vs
        va_mask = seasons == vs
        if va_mask.sum() == 0: continue

        tr_X = torch.FloatTensor(X[tr_mask]).to(DEVICE)
        tr_y = torch.FloatTensor(y[tr_mask]).to(DEVICE)
        va_X = torch.FloatTensor(X[va_mask]).to(DEVICE)
        va_y = y[va_mask]

        model = WideAndDeep(X.shape[1], deep_dims=[128, 64, 32]).to(DEVICE)
        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
        dataset = TensorDataset(tr_X, tr_y)
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

        best_loss, best_state, no_improve = 1.0, None, 0
        for epoch in range(epochs):
            model.train()
            for bx, by in loader:
                optimizer.zero_grad()
                pred = model(bx)
                loss = nn.MSELoss()(pred, by)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            scheduler.step()

            model.eval()
            with torch.no_grad():
                val_pred = model(va_X).cpu().numpy()
            val_pred = np.clip(val_pred, CLIP_MIN, CLIP_MAX)
            val_bs = np.mean((va_y - val_pred) ** 2)

            if val_bs < best_loss:
                best_loss = val_bs
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                no_improve = 0
            else:
                no_improve += 1
            if no_improve >= patience: break

        model.load_state_dict(best_state)
        model.eval()
        with torch.no_grad():
            final_pred = np.clip(model(va_X).cpu().numpy(), CLIP_MIN, CLIP_MAX)
        oof[va_mask] = final_pred

    valid = ~np.isnan(oof)
    bs = np.mean((y[valid] - oof[valid]) ** 2)
    print(f"  {name} Brier: {bs:.4f}")
    return oof, valid, bs

oof_wd, vm_wd, bs_wd = wide_deep_cv(tab_scaled, y_all, seasons_all, name="Wide&Deep")

### 3.10 Sklearn MLP (Simple Neural Net Baseline)

In [ ]:
print("\n" + "=" * 60)
print("MODEL 10: SKLEARN MLP")
print("=" * 60)

def sklearn_mlp_cv(X, y, seasons, name="SkMLP"):
    val_seasons = sorted(set(s for s in np.unique(seasons) if s >= 2015))
    oof = np.full(len(y), np.nan)
    for vs in val_seasons:
        tr, va = seasons < vs, seasons == vs
        if va.sum() == 0: continue
        m = MLPClassifier(hidden_layer_sizes=(128, 64, 32), activation='relu',
                           solver='adam', alpha=1e-3, batch_size=128,
                           learning_rate='adaptive', learning_rate_init=1e-3,
                           max_iter=200, early_stopping=True, n_iter_no_change=15,
                           validation_fraction=0.15, random_state=SEED)
        m.fit(X[tr], y[tr])
        oof[va] = np.clip(m.predict_proba(X[va])[:, 1], CLIP_MIN, CLIP_MAX)
    valid = ~np.isnan(oof)
    bs = np.mean((y[valid] - oof[valid]) ** 2)
    print(f"  {name} Brier: {bs:.4f}")
    return oof, valid, bs

oof_skmlp, vm_skmlp, bs_skmlp = sklearn_mlp_cv(tab_scaled, y_all, seasons_all, "Sklearn-MLP")

### 3.11 Elastic Net Logistic Regression

In [ ]:
print("\n" + "=" * 60)
print("MODEL 11: ELASTIC NET LR")
print("=" * 60)

def elastic_cv(X, y, seasons, name="ElasticNet"):
    from sklearn.linear_model import SGDClassifier
    val_seasons = sorted(set(s for s in np.unique(seasons) if s >= 2015))
    oof = np.full(len(y), np.nan)
    for vs in val_seasons:
        tr, va = seasons < vs, seasons == vs
        if va.sum() == 0: continue
        m = SGDClassifier(loss='log_loss', penalty='elasticnet', alpha=1e-4,
                           l1_ratio=0.5, max_iter=1000, random_state=SEED)
        m.fit(X[tr], y[tr])
        oof[va] = np.clip(m.predict_proba(X[va])[:, 1], CLIP_MIN, CLIP_MAX)
    valid = ~np.isnan(oof)
    bs = np.mean((y[valid] - oof[valid]) ** 2)
    print(f"  {name} Brier: {bs:.4f}")
    return oof, valid, bs

oof_enet, vm_enet, bs_enet = elastic_cv(tab_scaled, y_all, seasons_all, "ElasticNet-LR")

### 3.12 Gaussian Process Classifier (Uncertainty-Aware)

In [ ]:
print("\n" + "=" * 60)
print("MODEL 12: GAUSSIAN PROCESS")
print("=" * 60)

# GP is expensive - use subset of features for speed
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF, ConstantKernel

def gp_cv(X, y, seasons, name="GP"):
    # Use only top features to keep GP tractable
    # Select features with highest variance
    feat_var = np.var(X, axis=0)
    top_k = min(10, X.shape[1])
    top_idx = np.argsort(feat_var)[-top_k:]
    X_sub = X[:, top_idx]

    val_seasons = sorted(set(s for s in np.unique(seasons) if s >= 2015))
    oof = np.full(len(y), np.nan)
    for vs in val_seasons:
        tr, va = seasons < vs, seasons == vs
        if va.sum() == 0: continue
        kernel = ConstantKernel(1.0) * RBF(length_scale=1.0)
        m = GaussianProcessClassifier(kernel=kernel, max_iter_predict=100,
                                        n_restarts_optimizer=2, random_state=SEED)
        # Subsample training if too large (GP is O(n^3))
        tr_idx = np.where(tr)[0]
        if len(tr_idx) > 1000:
            tr_idx = np.random.choice(tr_idx, 1000, replace=False)
        m.fit(X_sub[tr_idx], y[tr_idx])
        oof[va] = np.clip(m.predict_proba(X_sub[va])[:, 1], CLIP_MIN, CLIP_MAX)
    valid = ~np.isnan(oof)
    bs = np.mean((y[valid] - oof[valid]) ** 2)
    print(f"  {name} Brier: {bs:.4f}")
    return oof, valid, bs

oof_gp, vm_gp, bs_gp = gp_cv(tab_scaled, y_all, seasons_all, "GaussianProcess")

## 4. Historical Seed Prior Model (Analytical Baseline)

In [ ]:
print("\n" + "=" * 60)
print("MODEL 13: HISTORICAL SEED PRIOR")
print("=" * 60)

# Compute historical win rates by seed matchup
def build_seed_lookup(tourney_df, seeds_df):
    """Build empirical win probability table indexed by (seed_a, seed_b)."""
    lookup = {}
    for _, g in tourney_df.iterrows():
        s = g['Season']
        w, l = g['WTeamID'], g['LTeamID']
        ws = seeds_df[(seeds_df['Season'] == s) & (seeds_df['TeamID'] == w)]
        ls = seeds_df[(seeds_df['Season'] == s) & (seeds_df['TeamID'] == l)]
        if len(ws) == 0 or len(ls) == 0: continue
        sw, sl = ws.iloc[0]['SeedNum'], ls.iloc[0]['SeedNum']
        ta, tb = min(w, l), max(w, l)
        sa, sb = (sw, sl) if ta == w else (sl, sw)
        key = (sa, sb)
        if key not in lookup:
            lookup[key] = {'wins': 0, 'total': 0}
        lookup[key]['total'] += 1
        if ta == w:
            lookup[key]['wins'] += 1
    # Convert to probabilities
    for key in lookup:
        n = lookup[key]['total']
        lookup[key]['prob'] = lookup[key]['wins'] / n if n > 0 else 0.5
    return lookup

def seed_prior_cv(meta_df, y, seasons, tourney_df, seeds_df, name="SeedPrior"):
    val_seasons = sorted(set(s for s in np.unique(seasons) if s >= 2015))
    oof = np.full(len(y), np.nan)

    for vs in val_seasons:
        # Build lookup from prior years only
        prior_tourney = tourney_df[tourney_df['Season'] < vs]
        lookup = build_seed_lookup(prior_tourney, seeds_df)

        va_mask = seasons == vs
        for i in np.where(va_mask)[0]:
            row = meta_df.iloc[i]
            sa = row.get('SeedA', 8)
            sb = row.get('SeedB', 8)
            key = (int(sa), int(sb))
            if key in lookup:
                oof[i] = np.clip(lookup[key]['prob'], CLIP_MIN, CLIP_MAX)
            else:
                # Fallback: logistic function of seed diff
                diff = sa - sb
                oof[i] = np.clip(1.0 / (1.0 + 10.0 ** (diff * 0.15)), CLIP_MIN, CLIP_MAX)

    valid = ~np.isnan(oof)
    bs = np.mean((y[valid] - oof[valid]) ** 2)
    print(f"  {name} Brier: {bs:.4f}")
    return oof, valid, bs

# Men's seed prior
m_seed_oof, _, m_seed_bs = seed_prior_cv(
    m_meta, m_y, m_meta['Season'].values,
    m_tourney_compact, m_seeds, "M-SeedPrior")

# Women's seed prior
w_seed_oof, _, w_seed_bs = seed_prior_cv(
    w_meta, w_y, w_meta['Season'].values,
    w_tourney_compact, w_seeds, "W-SeedPrior")

# Combine
oof_seedprior = np.concatenate([m_seed_oof, w_seed_oof])
vm_seedprior = ~np.isnan(oof_seedprior)
bs_seedprior = np.mean((y_all[vm_seedprior] - oof_seedprior[vm_seedprior]) ** 2)
print(f"  Combined SeedPrior Brier: {bs_seedprior:.4f}")

## 5. Expert Ensemble

In [ ]:
print("\n" + "=" * 60)
print("EXPERT ENSEMBLE")
print("=" * 60)

# Collect all OOF predictions
all_oof = {
    'BradleyTerry': bt_oof,
    'RandomForest': oof_rf,
    'ExtraTrees': oof_et,
    'SVM-RBF': oof_svm,
    'KNN': oof_knn,
    'GradientBoosting': oof_gbc,
    'BayesianRidge': oof_bayes,
    'Wide&Deep': oof_wd,
    'Sklearn-MLP': oof_skmlp,
    'ElasticNet-LR': oof_enet,
    'GaussianProcess': oof_gp,
    'SeedPrior': oof_seedprior,
}
if HAS_TABNET:
    all_oof['TabNet'] = oof_tabnet

# Find common valid mask
common_valid = np.ones(len(y_all), dtype=bool)
for name, oof in all_oof.items():
    common_valid &= ~np.isnan(oof)
print(f"Common valid predictions: {common_valid.sum()}")
y_valid = y_all[common_valid]

# Individual model scores
print("\nIndividual Model Scores:")
model_scores = {}
for name, oof in sorted(all_oof.items(), key=lambda x: np.mean((y_valid - x[1][common_valid]) ** 2)):
    bs = np.mean((y_valid - oof[common_valid]) ** 2)
    model_scores[name] = bs
    print(f"  {name:25s}: Brier={bs:.4f}")

# Optimize ensemble weights
names = list(all_oof.keys())
preds_matrix = np.column_stack([all_oof[n][common_valid] for n in names])

def ensemble_objective(weights):
    w = weights / weights.sum()
    ens = np.clip(preds_matrix @ w, CLIP_MIN, CLIP_MAX)
    return np.mean((y_valid - ens) ** 2)

n_models = len(names)
x0 = np.ones(n_models) / n_models
bounds = [(0, 1)] * n_models
constraints = {'type': 'eq', 'fun': lambda w: w.sum() - 1.0}
result = minimize(ensemble_objective, x0, bounds=bounds, constraints=constraints, method='SLSQP')

opt_weights = dict(zip(names, result.x))
opt_brier = result.fun

print(f"\nOptimized Expert Ensemble Brier: {opt_brier:.4f}")
print("\nOptimal Weights:")
for name, w in sorted(opt_weights.items(), key=lambda x: -x[1]):
    if w > 0.01:
        print(f"  {name:25s}: {w:.3f}")

# Simple average
simple_ens = np.clip(preds_matrix.mean(axis=1), CLIP_MIN, CLIP_MAX)
simple_brier = np.mean((y_valid - simple_ens) ** 2)
print(f"\nSimple Average Brier: {simple_brier:.4f}")

### 5.1 Stacking Meta-Learner (Ridge on Expert OOF)

In [ ]:
print("\nStacking Meta-Learner:")
meta_X = preds_matrix
meta_y = y_valid
meta_seasons = seasons_all[common_valid]
meta_oof = np.full(len(meta_y), np.nan)

for vs in sorted(set(s for s in np.unique(meta_seasons) if s >= 2018)):
    tr = meta_seasons < vs
    va = meta_seasons == vs
    if va.sum() == 0: continue
    meta_model = Ridge(alpha=1.0)
    meta_model.fit(meta_X[tr], meta_y[tr])
    meta_oof[va] = np.clip(meta_model.predict(meta_X[va]), CLIP_MIN, CLIP_MAX)

meta_valid = ~np.isnan(meta_oof)
if meta_valid.sum() > 0:
    stacking_brier = np.mean((meta_y[meta_valid] - meta_oof[meta_valid]) ** 2)
    print(f"  Expert Stacking Brier: {stacking_brier:.4f}")

### 5.2 Calibration Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Top models calibration
for name in ['BradleyTerry', 'RandomForest', 'Wide&Deep', 'GradientBoosting', 'SVM-RBF']:
    if name in all_oof:
        p = all_oof[name][common_valid]
        frac, mean_p = calibration_curve(y_valid, p, n_bins=10)
        axes[0].plot(mean_p, frac, 's-', label=name, markersize=4)
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[0].set_title('Expert Model Calibration'); axes[0].legend(fontsize=8)
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Observed')

# Ensemble calibration
w_arr = np.array([opt_weights[n] for n in names])
ens_pred = np.clip(preds_matrix @ w_arr, CLIP_MIN, CLIP_MAX)
frac, mean_p = calibration_curve(y_valid, ens_pred, n_bins=10)
axes[1].plot(mean_p, frac, 's-', label='Expert Ensemble', linewidth=2, color='red')
if meta_valid.sum() > 0:
    frac2, mean_p2 = calibration_curve(meta_y[meta_valid], meta_oof[meta_valid], n_bins=10)
    axes[1].plot(mean_p2, frac2, 'o-', label='Stacking', linewidth=2, color='blue')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[1].set_title('Ensemble Calibration'); axes[1].legend()
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Observed')

plt.tight_layout()
plt.savefig(OUT_DIR / 'expert_calibration.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Inference & Submission

In [ ]:
print("\n" + "=" * 60)
print("INFERENCE PIPELINE")
print("=" * 60)

# Retrain best models on full data
print("[1/3] Retraining models on full data...")

# Random Forest
rf_final = RandomForestClassifier(n_estimators=500, max_depth=8, min_samples_leaf=10,
                                    max_features='sqrt', random_state=SEED, n_jobs=-1)
rf_final.fit(tab_all, y_all)

# Extra Trees
et_final = ExtraTreesClassifier(n_estimators=500, max_depth=10, min_samples_leaf=8,
                                  max_features='sqrt', random_state=SEED, n_jobs=-1)
et_final.fit(tab_all, y_all)

# Gradient Boosting
gbc_final = GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=4,
                                         subsample=0.8, min_samples_leaf=10, random_state=SEED)
gbc_final.fit(tab_all, y_all)

# SVM
svm_final = SVC(C=1.0, kernel='rbf', gamma='scale', probability=True, random_state=SEED)
svm_final.fit(tab_scaled, y_all)

# KNN
knn_final = KNeighborsClassifier(n_neighbors=50, weights='distance')
knn_final.fit(tab_scaled, y_all)

# Bayesian Ridge
bayes_final = BayesianRidge(max_iter=300)
bayes_final.fit(tab_scaled, y_all)

# Sklearn MLP
skmlp_final = MLPClassifier(hidden_layer_sizes=(128, 64, 32), activation='relu',
                              solver='adam', alpha=1e-3, batch_size=128,
                              learning_rate='adaptive', max_iter=200,
                              random_state=SEED)
skmlp_final.fit(tab_scaled, y_all)

# Elastic Net
from sklearn.linear_model import SGDClassifier
enet_final = SGDClassifier(loss='log_loss', penalty='elasticnet', alpha=1e-4,
                            l1_ratio=0.5, max_iter=1000, random_state=SEED)
enet_final.fit(tab_scaled, y_all)

# Wide & Deep (full retrain)
wd_final = WideAndDeep(N_TAB_FEATURES, deep_dims=[128, 64, 32]).to(DEVICE)
wd_optimizer = optim.AdamW(wd_final.parameters(), lr=1e-3, weight_decay=1e-4)
wd_scheduler = optim.lr_scheduler.CosineAnnealingLR(wd_optimizer, T_max=100)
wd_dataset = TensorDataset(torch.FloatTensor(tab_scaled).to(DEVICE),
                             torch.FloatTensor(y_all).to(DEVICE))
wd_loader = DataLoader(wd_dataset, batch_size=128, shuffle=True)
wd_final.train()
for epoch in range(100):
    for bx, by in wd_loader:
        wd_optimizer.zero_grad()
        loss = nn.MSELoss()(wd_final(bx), by)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(wd_final.parameters(), 1.0)
        wd_optimizer.step()
    wd_scheduler.step()
wd_final.eval()

# Bradley-Terry (full)
bt_m_final = BradleyTerryModel(alpha=0.5, max_iter=150, lr=0.02)
bt_m_final.fit(pd.concat([m_reg_compact, m_tourney_compact], ignore_index=True))
bt_w_final = BradleyTerryModel(alpha=0.5, max_iter=150, lr=0.02)
bt_w_final.fit(pd.concat([w_reg_compact, w_tourney_compact], ignore_index=True))

# TabNet (full)
if HAS_TABNET:
    tabnet_final = TabNetClassifier(n_d=16, n_a=16, n_steps=3, gamma=1.5,
                                      lambda_sparse=1e-3, verbose=0, seed=SEED)
    tabnet_final.fit(tab_all, y_all.astype(int), max_epochs=100, patience=20,
                       batch_size=256, virtual_batch_size=128)

# Seed prior lookup (all data)
m_seed_lookup = build_seed_lookup(m_tourney_compact, m_seeds)
w_seed_lookup = build_seed_lookup(w_tourney_compact, w_seeds)

# Stacking meta-learner (full)
meta_model_final = Ridge(alpha=1.0)
meta_model_final.fit(meta_X, meta_y)

print("  All models retrained.")

In [ ]:
# ========== PREDICT FUNCTION ==========
all_seeds = pd.concat([m_seeds, w_seeds], ignore_index=True)
all_elo_all = pd.concat([m_elo_df, w_elo_df], ignore_index=True)
all_stats = pd.concat([m_stats, w_stats], ignore_index=True)

def predict_matchup_expert(season, team_a, team_b):
    """Expert ensemble prediction for a single matchup."""

    vec_a = get_team_vector(all_stats, all_elo_all, season, team_a)
    vec_b = get_team_vector(all_stats, all_elo_all, season, team_b)
    if vec_a is None: vec_a = np.zeros(N_TEAM_FEATURES, dtype=np.float32)
    if vec_b is None: vec_b = np.zeros(N_TEAM_FEATURES, dtype=np.float32)

    # Seeds
    sa = all_seeds[(all_seeds['Season'] == season) & (all_seeds['TeamID'] == team_a)]
    sb = all_seeds[(all_seeds['Season'] == season) & (all_seeds['TeamID'] == team_b)]
    seed_a = sa.iloc[0]['SeedNum'] if len(sa) > 0 else 8
    seed_b = sb.iloc[0]['SeedNum'] if len(sb) > 0 else 8

    # Build tabular features (same as training)
    diff = vec_a - vec_b
    tab = np.concatenate([diff, [seed_a - seed_b, seed_a, seed_b]])

    # Massey
    is_mens = 1000 <= team_a <= 1999
    massey_feats = []
    if is_mens:
        am = m_massey_feat[(m_massey_feat['Season'] == season) & (m_massey_feat['TeamID'] == team_a)]
        bm = m_massey_feat[(m_massey_feat['Season'] == season) & (m_massey_feat['TeamID'] == team_b)]
        for sys_name in ['POM', 'SAG', 'MOR', 'ConsensusRank']:
            if len(am) > 0 and len(bm) > 0 and sys_name in am.columns:
                va_val = am.iloc[0][sys_name] if not pd.isna(am.iloc[0].get(sys_name)) else 150
                vb_val = bm.iloc[0][sys_name] if not pd.isna(bm.iloc[0].get(sys_name)) else 150
                massey_feats.append(va_val - vb_val)
            else:
                massey_feats.append(0)
    else:
        massey_feats = [0, 0, 0, 0]
    tab = np.concatenate([tab, massey_feats])

    # Coach
    coach_feats = [0]
    if is_mens and 'm_coach_feat' in dir():
        ca = m_coach_feat[(m_coach_feat['Season'] == season) & (m_coach_feat['TeamID'] == team_a)]
        cb = m_coach_feat[(m_coach_feat['Season'] == season) & (m_coach_feat['TeamID'] == team_b)]
        if len(ca) > 0 and len(cb) > 0:
            coach_feats = [ca.iloc[0]['CoachTourneyWins'] - cb.iloc[0]['CoachTourneyWins']]
    tab = np.concatenate([tab, coach_feats])

    # Conf tourney
    conf_feats = [0]
    if is_mens and HAS_CONF:
        cta = m_conf_feat[(m_conf_feat['Season'] == season) & (m_conf_feat['TeamID'] == team_a)]
        ctb = m_conf_feat[(m_conf_feat['Season'] == season) & (m_conf_feat['TeamID'] == team_b)]
        if len(cta) > 0 and len(ctb) > 0:
            conf_feats = [cta.iloc[0]['ConfTourneyWins'] - ctb.iloc[0]['ConfTourneyWins']]
    tab = np.concatenate([tab, conf_feats])

    # Interactions
    elo_diff = vec_a[-1] - vec_b[-1]
    net_idx = TEAM_FEATURES.index('NetRating')
    net_diff = diff[net_idx]
    seed_diff = seed_a - seed_b
    tab = np.concatenate([tab, [seed_diff * elo_diff, seed_diff * net_diff]])

    # Quadratic seed
    tab = np.concatenate([tab, [seed_diff ** 2, abs(seed_diff)]])

    # Pad/trim
    if len(tab) < N_TAB_FEATURES:
        tab = np.pad(tab, (0, N_TAB_FEATURES - len(tab)))
    tab = tab[:N_TAB_FEATURES]
    tab = np.nan_to_num(tab, nan=0.0).reshape(1, -1).astype(np.float32)
    tab_s = tab_scaler.transform(tab)

    preds = {}

    # Traditional ML
    preds['RandomForest'] = rf_final.predict_proba(tab)[0, 1]
    preds['ExtraTrees'] = et_final.predict_proba(tab)[0, 1]
    preds['GradientBoosting'] = gbc_final.predict_proba(tab)[0, 1]
    preds['SVM-RBF'] = svm_final.predict_proba(tab_s)[0, 1]
    preds['KNN'] = knn_final.predict_proba(tab_s)[0, 1]
    preds['BayesianRidge'] = np.clip(bayes_final.predict(tab_s)[0], CLIP_MIN, CLIP_MAX)
    preds['Sklearn-MLP'] = skmlp_final.predict_proba(tab_s)[0, 1]
    preds['ElasticNet-LR'] = enet_final.predict_proba(tab_s)[0, 1]

    # Bradley-Terry
    bt_model = bt_m_final if is_mens else bt_w_final
    preds['BradleyTerry'] = bt_model.predict(team_a, team_b)

    # Wide & Deep
    with torch.no_grad():
        wd_pred = wd_final(torch.FloatTensor(tab_s).to(DEVICE)).cpu().item()
    preds['Wide&Deep'] = wd_pred

    # TabNet
    if HAS_TABNET:
        preds['TabNet'] = tabnet_final.predict_proba(tab)[0, 1]

    # Seed prior
    lookup = m_seed_lookup if is_mens else w_seed_lookup
    key = (int(seed_a), int(seed_b))
    if key in lookup:
        preds['SeedPrior'] = lookup[key]['prob']
    else:
        preds['SeedPrior'] = 1.0 / (1.0 + 10.0 ** (seed_diff * 0.15))

    # Gaussian Process (skip for speed in inference - too slow)
    preds['GaussianProcess'] = preds['BayesianRidge']  # Use Bayesian Ridge as proxy

    # Weighted ensemble
    final_pred = sum(opt_weights.get(name, 0) * np.clip(p, CLIP_MIN, CLIP_MAX) for name, p in preds.items())
    total_w = sum(opt_weights.get(name, 0) for name in preds if name in opt_weights)
    if total_w > 0:
        final_pred /= total_w

    return np.clip(final_pred, CLIP_MIN, CLIP_MAX)

In [ ]:
# ========== GENERATE SUBMISSIONS ==========
print("\n[2/3] Generating submissions...")

def generate_submission(sub_df, filename):
    predictions = []
    for i, row in sub_df.iterrows():
        parts = row['ID'].split('_')
        season, ta, tb = int(parts[0]), int(parts[1]), int(parts[2])
        pred = predict_matchup_expert(season, ta, tb)
        predictions.append(pred)
        if (i + 1) % 50000 == 0:
            print(f"    {i+1}/{len(sub_df)} predictions done...")

    sub_df = sub_df.copy()
    sub_df['Pred'] = predictions
    sub_df.to_csv(OUT_DIR / filename, index=False)
    preds_arr = np.array(predictions)
    print(f"  Saved {filename}: {len(sub_df)} rows, mean={preds_arr.mean():.4f}, "
          f"std={preds_arr.std():.4f}, range=[{preds_arr.min():.4f}, {preds_arr.max():.4f}]")
    return sub_df

sub1_expert = generate_submission(sub1, 'submission_stage1_expert.csv')
sub2_expert = generate_submission(sub2, 'submission_stage2_expert.csv')

# Conservative blend
print("\n[3/3] Creating conservative submission...")
def seed_prior_func(ta, tb, season):
    sa = all_seeds[(all_seeds['Season'] == season) & (all_seeds['TeamID'] == ta)]
    sb = all_seeds[(all_seeds['Season'] == season) & (all_seeds['TeamID'] == tb)]
    if len(sa) > 0 and len(sb) > 0:
        diff = sa.iloc[0]['SeedNum'] - sb.iloc[0]['SeedNum']
        return 1.0 / (1.0 + 10.0 ** (diff * 0.15))
    return 0.5

seed_preds = []
for _, row in sub2.iterrows():
    parts = row['ID'].split('_')
    seed_preds.append(seed_prior_func(int(parts[1]), int(parts[2]), int(parts[0])))
seed_preds = np.array(seed_preds)

model_preds = sub2_expert['Pred'].values
conservative = np.clip(0.20 * seed_preds + 0.80 * model_preds, CLIP_MIN, CLIP_MAX)
sub2_cons = sub2.copy()
sub2_cons['Pred'] = conservative
sub2_cons.to_csv(OUT_DIR / 'submission_stage2_expert_conservative.csv', index=False)
print(f"  Conservative: mean={conservative.mean():.4f}, std={conservative.std():.4f}")

## 7. Final Summary

In [ ]:
print("\n" + "=" * 60)
print("EXPERT MODELS - FINAL RESULTS")
print("=" * 60)

print("\nModel Performance (Leave-One-Season-Out CV, Brier Score):")
print("-" * 55)
all_scores = {
    'BradleyTerry': bt_brier,
    'RandomForest': bs_rf,
    'ExtraTrees': bs_et,
    'SVM-RBF': bs_svm,
    'KNN': bs_knn,
    'GradientBoosting': bs_gbc,
    'BayesianRidge': bs_bayes,
    'Wide&Deep': bs_wd,
    'Sklearn-MLP': bs_skmlp,
    'ElasticNet-LR': bs_enet,
    'GaussianProcess': bs_gp,
    'SeedPrior': bs_seedprior,
}
if HAS_TABNET:
    all_scores['TabNet'] = bs_tabnet

for name, bs in sorted(all_scores.items(), key=lambda x: x[1]):
    bar = '|' * int((0.25 - bs) * 200)
    print(f"  {name:25s}: {bs:.4f} {bar}")

print(f"\n  {'EXPERT ENSEMBLE':25s}: {opt_brier:.4f} ***")
print(f"  {'Simple Average':25s}: {simple_brier:.4f}")
if meta_valid.sum() > 0:
    print(f"  {'Stacking':25s}: {stacking_brier:.4f}")

print(f"\nEnsemble Weights (non-zero):")
for name, w in sorted(opt_weights.items(), key=lambda x: -x[1]):
    if w > 0.01:
        print(f"  {name:25s}: {w:.1%}")

print(f"\nSubmissions generated:")
print(f"  {OUT_DIR / 'submission_stage1_expert.csv'}")
print(f"  {OUT_DIR / 'submission_stage2_expert.csv'} (aggressive)")
print(f"  {OUT_DIR / 'submission_stage2_expert_conservative.csv'} (conservative)")
print(f"\nUpload these to Kaggle alongside your other notebook submissions.")